# Exploratory Data Analysis — GridForecast (Spain)

This notebook explores the **raw (bronze) layer** produced by `src/ingest.py`
(demand, generation by technology, and wind/solar forecast for Spain, at 15-min
resolution). The goal is to understand the data *before* cleaning and modeling:
seasonality of the target, the generation mix, forecast vs. actual for
renewables, and data-quality / alignment issues that the next (silver) layer
will need to handle.

Data source: ENTSO-E Transparency Platform (EU Regulation 543/2013).

## 1. Load the raw data

We read the latest Parquet file from each `data/raw/es/<type>/` folder. Using a
small helper keeps the notebook robust to the date range in each filename.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAW = Path("../data/raw/es")

def latest(subdir):
    files = sorted((RAW / subdir).glob("*.parquet"))
    if not files:
        raise FileNotFoundError(
            f"No parquet in {RAW/subdir}. Run `python src/ingest.py` first."
        )
    return pd.read_parquet(files[-1])

load = latest("load")
generation = latest("generation")
forecast = latest("wind_solar_forecast")

print("load:      ", load.shape)
print("generation:", generation.shape)
print("forecast:  ", forecast.shape)

## 2. The forecasting target: electricity demand

`Actual Load` is what we ultimately want to predict. First, the raw series over
the whole window.

In [ ]:
col = "Actual Load"
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(load.index, load[col], lw=1, color="#185FA5")
ax.set(title="Electricity demand — Spain", ylabel="MW")
plt.tight_layout(); plt.show()

### 2.1 Daily profile

Averaging by hour of day reveals the daily cycle (morning ramp, midday and
evening peaks, overnight valley). The shaded band shows the min–max range.

In [ ]:
by_hour = load[col].groupby(load.index.hour).agg(["mean", "min", "max"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(by_hour.index, by_hour["mean"], marker="o", color="#185FA5")
ax.fill_between(by_hour.index, by_hour["min"], by_hour["max"], alpha=0.15, color="#185FA5")
ax.set(title="Average daily demand profile", xlabel="hour of day", ylabel="MW")
plt.tight_layout(); plt.show()

### 2.2 Weekday vs. weekend

Demand patterns usually differ on weekends. With only one week of data this is
indicative, not conclusive (see the limitations note at the end).

In [ ]:
df = load.copy()
df["hour"] = df.index.hour
df["kind"] = ["weekend" if d >= 5 else "weekday" for d in df.index.dayofweek]
prof = df.groupby(["kind", "hour"])[col].mean().unstack("kind")
prof.plot(figsize=(8, 4), title="Demand profile: weekday vs. weekend")
plt.ylabel("MW"); plt.tight_layout(); plt.show()

### 2.3 Distribution and autocorrelation

The histogram shows the spread of demand values. The 1-day autocorrelation
quantifies how strongly demand repeats every 24 h (96 steps at 15-min
resolution) — a high value means the daily cycle is a strong, learnable signal.

In [ ]:
load[col].plot(kind="hist", bins=40, figsize=(7, 4), color="#185FA5",
               title="Demand distribution")
plt.xlabel("MW"); plt.tight_layout(); plt.show()

In [ ]:
lag_1d = 96  # 15-min steps in one day
print("Autocorrelation at 1-day lag:", round(load[col].autocorr(lag_1d), 3))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(load[col][:-lag_1d], load[col][lag_1d:], s=6, alpha=0.4, color="#185FA5")
ax.set(title="Demand vs. demand one day earlier", xlabel="t (MW)", ylabel="t + 1 day (MW)")
plt.tight_layout(); plt.show()

## 3. Generation mix

Several technologies are reported but are zero for Spain in this window. We drop
the all-zero columns and look at the average mix of what actually runs.

In [ ]:
nonzero = generation.loc[:, (generation != 0).any()]
zero_cols = [c for c in generation.columns if c not in nonzero.columns]
print("All-zero columns dropped:", zero_cols)

nonzero.mean().sort_values().plot(kind="barh", figsize=(8, 6),
                                  title="Average generation by source (MW)")
plt.xlabel("MW"); plt.tight_layout(); plt.show()

### 3.1 Main sources over time

Nuclear runs as a flat baseload, fossil gas is the flexible source that follows
demand, solar has a strong daily cycle, and wind is variable.

In [ ]:
key = [
    "Nuclear - Actual Aggregated",
    "Fossil Gas - Actual Aggregated",
    "Solar - Actual Aggregated",
    "Wind Onshore - Actual Aggregated",
    "Hydro Water Reservoir - Actual Aggregated",
]
key = [c for c in key if c in generation.columns]
generation[key].plot(figsize=(11, 4), title="Main generation sources over time")
plt.ylabel("MW"); plt.tight_layout(); plt.show()

## 4. Renewables: forecast vs. actual

The forecast file carries ENTSO-E's day-ahead Solar and Wind Onshore forecasts.
Comparing them to the actual generation gives us a ready-made **baseline** to
beat when we model, plus a sense of how much error and bias the official
forecast already has.

In [ ]:
comp = pd.DataFrame({
    "solar_actual":   generation["Solar - Actual Aggregated"],
    "solar_forecast": forecast["Solar"],
    "wind_actual":    generation["Wind Onshore - Actual Aggregated"],
    "wind_forecast":  forecast["Wind Onshore"],
}).dropna()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
comp[["solar_actual", "solar_forecast"]].plot(ax=axes[0], title="Solar: forecast vs. actual")
axes[0].set_ylabel("MW")
comp[["wind_actual", "wind_forecast"]].plot(ax=axes[1], title="Wind onshore: forecast vs. actual")
axes[1].set_ylabel("MW")
plt.tight_layout(); plt.show()

In [ ]:
for v in ["solar", "wind"]:
    err = comp[f"{v}_forecast"] - comp[f"{v}_actual"]
    print(f"{v:5}  MAE = {err.abs().mean():6.0f} MW   bias = {err.mean():+6.0f} MW")

## 5. Data quality and alignment

Before building the silver layer we check: row counts, time coverage, missing
values, and how well the three sources line up on their timestamps.

In [ ]:
for name, d in [("load", load), ("generation", generation), ("forecast", forecast)]:
    print(f"{name:11} rows={len(d):4}  {d.index.min()}  ->  {d.index.max()}  "
          f"nulls={int(d.isna().sum().sum())}")

joined = (load[["Actual Load"]]
          .join(generation, how="inner")
          .join(forecast, how="inner", rsuffix="_forecast"))
print("\nRows after inner join on timestamp:", len(joined))

## 6. Takeaways

- **Target is highly seasonal.** Demand shows a strong, regular daily cycle
  (high 1-day autocorrelation) — a very learnable signal for forecasting.
- **Generation mix is sparse.** Many technology columns are all-zero for Spain;
  the silver layer should drop them. The live sources are nuclear (baseload),
  fossil gas (flexible), solar (daily cycle) and wind (variable).
- **A baseline already exists.** ENTSO-E's wind/solar forecast vs. actual gives
  us an error/bias benchmark our own model should aim to beat.
- **Alignment needed.** The three sources have slightly different lengths and
  end times, so the silver layer must join them on a common, regular timestamp
  index and handle the edges.

**Limitation:** this is only ~7 days of data, so weekly seasonality and holiday
effects can't be measured yet. A recurring ingestion that accumulates history
(a later MLOps task) will unlock that.

**Next step:** build the *silver* layer — a cleaned, timestamp-aligned table
combining demand + generation + forecast, ready for modeling.